# 41-1 Exploración de Texto 2 — Análisis de PDFs con PySpark

Este notebook demuestra cómo extraer texto de archivos PDF, tokenizarlo y analizarlo usando únicamente funciones nativas de PySpark, sin dependencias externas de NLP.

## Objetivo
Leer archivos PDF desde una carpeta local, extraer su contenido, tokenizar el texto, calcular frecuencias de palabras (global y por documento) y buscar términos específicos.

## Flujo del notebook
1. Instalación de dependencias (visualización y lectura de PDFs).
2. Importación de módulos PySpark y PyPDF2.
3. Creación de la sesión Spark.
4. Extracción de texto desde archivos PDF.
5. Tokenización y normalización con funciones nativas de PySpark.
6. Conteo de frecuencia de palabras (global).
7. Top 5 palabras más frecuentes por documento.
8. Búsqueda de un término específico en los documentos.

## Antes de ejecutar
- Verifica que la sesión de Spark esté activa y sin errores.
- Confirma que la carpeta `pdf/` contenga archivos PDF.
- Ejecuta las celdas en orden para mantener dependencias.


## 1. Instalación de dependencias
Se instalan librerías para visualización (`matplotlib`, `wordcloud`), manejo de datos (`numpy`, `pandas`) y lectura de PDFs (`pypdf2`). No se requiere ninguna librería externa de NLP.


In [ ]:
%pip install numpy pandas matplotlib seaborn wordcloud pypdf2

## 2. Importación de módulos
Se importan los módulos de PySpark (sesión, tipos, funciones), `PyPDF2` para leer PDFs y `os` para recorrer archivos en la carpeta.


In [ ]:
import findspark
findspark.init('/Users/joseaguilar/Documents/Development/spark/spark-3.5.1-bin-hadoop3')
from pyspark import *
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType,StructField, StringType, IntegerType, DateType, TimestampType, LongType
from pyspark.sql.types import ArrayType, DoubleType, BooleanType, DecimalType
from pyspark.sql.functions import regexp_extract, split, from_unixtime, col, avg, min, max
from pyspark.sql.functions import grouping_id, window, explode, to_json, from_json
from pyspark.sql.functions import udf, lit, current_timestamp, current_date, date_format
import PyPDF2
import os

## 3. Creación de la sesión Spark
Se inicializa una `SparkSession` básica sin dependencias externas de NLP.


In [ ]:
spark = SparkSession.builder \
    .appName("exploracion_texto_pdf") \
    .getOrCreate()

## 4. Importaciones adicionales para procesamiento de texto
Se importan las funciones nativas de PySpark que se usarán para tokenización y normalización: `lower`, `regexp_replace`, `trim`, `split`, `explode`.


In [ ]:
from pyspark.sql.functions import lower, regexp_replace, trim, split, explode, col, size

print(f"Spark version: {spark.version}")
import sys
print(f"Python version: {sys.version}")

## 5. Extracción de texto desde archivos PDF
Se recorren todos los archivos `.pdf` de la carpeta `pdf/`, se extrae el texto de cada página con `PyPDF2` y se crea un DataFrame de Spark con columnas `documento` y `texto`.

**Sugerencia:** Verifica que la carpeta `pdf/` contenga archivos y ajusta la ruta si es necesario.


In [ ]:
textos_pdf = []
for nombre_archivo in os.listdir("/Users/joseaguilar/Development/spark_databricks/pdf"):
    if nombre_archivo.endswith(".pdf"):
        ruta = os.path.join("pdf", nombre_archivo)
        print(f"Procesando archivo: {ruta}")
        reader = PyPDF2.PdfReader(ruta)
        # Extraer todo el texto concatenando páginas
        texto = ""
        for page in reader.pages:
            texto += page.extract_text() + " "
        textos_pdf.append((nombre_archivo, texto))

# Crear un DataFrame de Spark con columnas 'documento' y 'texto'
df_docs = spark.createDataFrame(textos_pdf, schema=["documento", "texto"])
df_docs.show(1)

## 6. Tokenización y normalización con PySpark nativo
Se reemplaza la pipeline de Spark NLP por funciones nativas de PySpark:
- `lower()` → convierte a minúsculas.
- `regexp_replace()` → elimina caracteres no deseados (puntuación, símbolos).
- `trim()` → elimina espacios sobrantes.
- `split()` → divide el texto en tokens (palabras) por espacios.

El resultado es una columna `palabras` con un array de tokens por cada documento.


In [ ]:
# Tokenización y normalización usando funciones nativas de PySpark
# 1. Convertir a minúsculas
df_clean = df_docs.withColumn("texto_limpio", lower(col("texto")))

# 2. Eliminar caracteres no deseados (mantener letras, números, acentos, ñ y espacios)
df_clean = df_clean.withColumn(
    "texto_limpio",
    regexp_replace(col("texto_limpio"), r"[^\w\sáéíóúÁÉÍÓÚüÜñÑ¿?¡!]", "")
)

# 3. Reemplazar múltiples espacios por uno solo y hacer trim
df_clean = df_clean.withColumn(
    "texto_limpio",
    trim(regexp_replace(col("texto_limpio"), r"\s+", " "))
)

# 4. Dividir en tokens (palabras)
df_tokens_pdf = df_clean.withColumn("palabras", split(col("texto_limpio"), " ")) \
                        .select("documento", "palabras")

df_tokens_pdf.show(1, truncate=100)

## 7. Conteo global de frecuencia de palabras
Se hace `explode` de la columna de arrays para obtener una palabra por fila, se filtran tokens vacíos y se agrupan para contar la frecuencia global de cada palabra.


In [ ]:
# Explode de la columna de listas de palabras para tener una palabra por fila
from pyspark.sql.functions import explode, col
df_palabras = df_tokens_pdf.select(explode(col("palabras")).alias("palabra"))
# Filtrar posibles tokens vacíos o muy cortos si aplica
df_palabras = df_palabras.filter(col("palabra") != "")

# Contar frecuencia de cada palabra
df_freq = df_palabras.groupBy("palabra").count().orderBy(col("count").desc())
df_freq.show(10)

## 8. Top 5 palabras más frecuentes por documento
Se usa una ventana (`Window`) con `row_number()` particionada por documento para obtener las 5 palabras más frecuentes de cada PDF.


In [ ]:
# Explotar las palabras por documento y contar frecuencias dentro de cada documento
df_palabras_pdf = df_tokens_pdf.select(col("documento"), explode(col("palabras")).alias("palabra"))
df_freq_por_doc = df_palabras_pdf.groupBy("documento", "palabra").count()
# Por ejemplo, mostrar las 5 palabras más frecuentes de cada documento
from pyspark.sql.window import Window
import pyspark.sql.functions as F
window = Window.partitionBy("documento").orderBy(F.desc("count"))
df_top5_por_doc = df_freq_por_doc.withColumn("rank", F.row_number().over(window))\
                                 .filter(col("rank") <= 5)\
                                 .orderBy("documento", col("rank"))
df_top5_por_doc.show(20, truncate=False)

## 9. Búsqueda de un término específico
Se filtra la columna de palabras usando `rlike` para buscar ocurrencias de un término concreto en cada documento y se cuenta cuántas veces aparece por PDF.


In [ ]:
term = "attention"  # término a buscar
df_term_counts = df_palabras_pdf.filter(col("palabra").rlike(term))\
                                .groupBy("documento").count()
df_term_counts.show()